# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nooragab/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

#Setup

In [1]:
%pip install -q duckdb scikit-learn

import duckdb, os
import pandas as pd, numpy as np
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
print("DuckDB connected, HF secret set. Ready to query the warehouse.")

DuckDB connected, HF secret set. Ready to query the warehouse.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method Choice and Why

**Label redesign:** in w03/w04 I used a within-month proxy (first half vs second half of
March), which the contract flagged as weak. Here I use a genuine past→future label:
**features from March 2026, label = whether the page's April 2026 impressions fell below
its March impressions.** This removes the overlap-window weakness and matches the lane
guide's recommendation for a stronger target.

**Models chosen:** Logistic Regression (a simple, interpretable linear baseline model) and
Random Forest (captures non-linear interactions between staleness, position, and CTR gap).
This mirrors the starter pipeline, where Random Forest beat both the hand rule and Logistic
Regression on the same kind of ranking task — I want to confirm whether that holds on real
warehouse data too.

**Why not clustering or gradient boosting:** my lane is Refresh/Content Opportunity Scoring,
a ranking-via-classification problem with a clear binary label — clustering doesn't fit a
labeled ranking task, and gradient boosting is a reasonable next step but adds complexity I
want to justify only if Random Forest doesn't already clear the baseline.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Method check: confirming this is a binary classification task.")
print("Toolkit options considered: LogisticRegression, RandomForestClassifier")
print("Both output a probability -> used to rank pages, same as the baseline rule.")

Method check: confirming this is a binary classification task.
Toolkit options considered: LogisticRegression, RandomForestClassifier
Both output a probability -> used to rank pages, same as the baseline rule.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split Design

**Client-holdout split (grouped by `client_hash_id`).** Pages from the same client often
share patterns (a client's whole site might get updated together, like the 9-of-20
concentration I found in ML-07's top picks). If pages from one client land in both train and
test, the model could just memorize client-specific quirks instead of learning generalizable
signal. A `GroupShuffleSplit` keeps every client's pages entirely on one side. I also use a
past→future feature/label split (March features, April label) so the test is time-aware too.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

build_query = f"""
    WITH march AS (
        SELECT content_hash_id, client_hash_id,
               SUM(gsc_impressions) AS impressions_month,
               SUM(gsc_clicks) AS clicks_month,
               AVG(gsc_avg_position) AS avg_position_month
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id, client_hash_id
    ),
    april AS (
        SELECT content_hash_id,
               SUM(gsc_impressions) AS impressions_april
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-04/*.parquet')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    ),
    tiered AS (
        SELECT m.*, a.impressions_april,
            (a.impressions_april < m.impressions_month)::INT AS is_declining_next_month,
            m.clicks_month / NULLIF(m.impressions_month, 0) AS ctr_month,
            CASE
                WHEN m.avg_position_month <= 3 THEN '1_top_3'
                WHEN m.avg_position_month <= 10 THEN '2_striking_4_10'
                WHEN m.avg_position_month <= 20 THEN '3_page_2'
                ELSE '4_deep_20plus'
            END AS position_tier
        FROM march m
        JOIN april a ON m.content_hash_id = a.content_hash_id
        WHERE m.impressions_month >= 100
    ),
    tier_avg AS (
        SELECT position_tier, AVG(ctr_month) AS tier_avg_ctr FROM tiered GROUP BY position_tier
    )
    SELECT
        t.content_hash_id, t.client_hash_id, t.impressions_month, t.avg_position_month,
        t.position_tier, ROUND(t.ctr_month, 4) AS ctr_month,
        ROUND(ta.tier_avg_ctr, 4) AS tier_avg_ctr,
        ROUND(GREATEST(ta.tier_avg_ctr - t.ctr_month, 0), 4) AS ctr_gap,
        DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') AS days_since_last_update,
        (DATE_DIFF('day', d.content_updated_date, DATE '2026-03-31') >= 90)::INT AS is_stale,
        d.word_count, t.is_declining_next_month
    FROM tiered t
    JOIN tier_avg ta ON t.position_tier = ta.position_tier
    JOIN read_parquet('{rel}/dim_content.parquet') d ON t.content_hash_id = d.content_hash_id
"""

data = con.sql(build_query).df()
data["baseline_score"] = data["impressions_month"] * data["is_stale"] * data["ctr_gap"]
data["word_count"] = data["word_count"].fillna(data["word_count"].median())

print(f"Full dataset: {data.shape[0]:,} rows, {data['client_hash_id'].nunique()} clients")
print(f"Label balance: {data['is_declining_next_month'].mean():.3f} declining into April")
data.head(3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Full dataset: 100,893 rows, 43 clients
Label balance: 0.656 declining into April


,content_hash_id,client_hash_id,impressions_month,avg_position_month,position_tier,ctr_month,tier_avg_ctr,ctr_gap,days_since_last_update,is_stale,word_count,is_declining_next_month,baseline_score
0,content_04c67f3541177192,client_0797ff3a1fc9a6a5,331.0,14.129210,3_page_2,0.006,0.0024,0.0000,34,0,3168,0,0.0
1,content_0f30e04e709c7b5d,client_0797ff3a1fc9a6a5,145.0,8.470926,2_striking_4_10,0.000,0.0032,0.0032,34,0,3211,1,0.0
2,content_1207efddce873942,client_0797ff3a1fc9a6a5,461.0,14.859827,3_page_2,0.000,0.0024,0.0024,-50,0,3465,0,0.0


In [4]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(data, groups=data["client_hash_id"]))

train, test = data.iloc[train_idx].copy(), data.iloc[test_idx].copy()

overlap = set(train["client_hash_id"]) & set(test["client_hash_id"])
print(f"Train: {len(train):,} rows, {train['client_hash_id'].nunique()} clients")
print(f"Test:  {len(test):,} rows, {test['client_hash_id'].nunique()} clients")
print(f"Client overlap between train/test (should be 0): {len(overlap)}")

Train: 84,384 rows, 32 clients
Test:  16,509 rows, 11 clients
Client overlap between train/test (should be 0): 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## Training and Comparing to the Baseline

Same features used in the ML-07 baseline (impressions, position tier, CTR gap, staleness,
word count), fed into Logistic Regression and Random Forest. Evaluated with Precision@20 and
Precision@50 on the held-out test clients — the exact same metric family used for the
baseline in ML-07.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

feature_cols = ["impressions_month", "avg_position_month", "ctr_month",
                "tier_avg_ctr", "ctr_gap", "is_stale", "days_since_last_update", "word_count"]

X_train = pd.get_dummies(train[feature_cols + ["position_tier"]], columns=["position_tier"])
X_test = pd.get_dummies(test[feature_cols + ["position_tier"]], columns=["position_tier"])
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train, y_test = train["is_declining_next_month"], test["is_declining_next_month"]

logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42).fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42).fit(X_train, y_train)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

results = {}
for k in (20, 50):
    results[f"baseline_p{k}"] = precision_at_k(test["baseline_score"], y_test, k)
    results[f"logreg_p{k}"] = precision_at_k(logreg.predict_proba(X_test)[:, 1], y_test, k)
    results[f"rf_p{k}"] = precision_at_k(rf.predict_proba(X_test)[:, 1], y_test, k)

comparison = pd.DataFrame({
    "method": ["baseline_rule", "logistic_regression", "random_forest"],
    "Precision@20": [results["baseline_p20"], results["logreg_p20"], results["rf_p20"]],
    "Precision@50": [results["baseline_p50"], results["logreg_p50"], results["rf_p50"]],
})
comparison

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,method,Precision@20,Precision@50
0,baseline_rule,0.75,0.64
1,logistic_regression,0.70,0.74
2,random_forest,0.70,0.72


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Errors and Interpretation

**The comparison isn't a clean win for the model.** At Precision@20, the baseline rule
actually beats both models (0.75 vs 0.70 for both). Only at Precision@50 do the models pull
ahead (0.74 logistic regression, 0.72 random forest, vs 0.64 baseline). This means the
baseline is still sharper at the very top of the queue, while the models find more real
signal once you look deeper into the ranking — a more honest and more useful finding than
"the model wins."

**What the model actually leans on:** feature importance shows `avg_position_month` (0.26)
and `impressions_month` (0.26) as the two dominant signals, followed by `word_count` (0.20)
and `days_since_last_update` (0.12). Strikingly, `is_stale` — the binary flag my entire
baseline rule was built around — has almost zero importance (0.0003). The random forest
found that raw position and traffic volume carry far more predictive power than a simple
90-day staleness cutoff. This is a real, useful discovery: my hand-written rule leaned on
the wrong lever.

**Error balance:** false negatives (4,161 missed declines) and false positives (3,397 wrongly
flagged) are reasonably balanced — the model isn't systematically over- or under-flagging.
Looking at the false negatives sample, several have `ctr_gap = 0.0000` and `is_stale = 0`,
meaning the model correctly saw no CTR problem and no staleness — but the page still declined
anyway, likely for a reason not captured by any of my five features (e.g. a competitor
outranking it, or a seasonal shift).

**Takeaway:** the model doesn't just "beat" the baseline in a simple sense — it trades
top-of-queue precision for deeper-queue precision, and it revealed that staleness (my
baseline's core assumption) isn't actually the strongest signal in this data. That's a more
valuable result than a bigger number would have been.

In [6]:
importances = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)
importances

,feature,importance
1,avg_position_month,0.259791
0,impressions_month,0.257648
7,word_count,0.196379
6,days_since_last_update,0.120596
2,ctr_month,0.108006
4,ctr_gap,0.046401
3,tier_avg_ctr,0.004053
9,position_tier_2_striking_4_10,0.002424
10,position_tier_3_page_2,0.001684
11,position_tier_4_deep_20plus,0.001461


In [7]:
test_eval = test.copy()
test_eval["rf_prob"] = rf.predict_proba(X_test)[:, 1]
test_eval["rf_pred"] = (test_eval["rf_prob"] >= 0.5).astype(int)

false_negatives = test_eval[(test_eval["is_declining_next_month"] == 1) & (test_eval["rf_pred"] == 0)]
false_positives = test_eval[(test_eval["is_declining_next_month"] == 0) & (test_eval["rf_pred"] == 1)]

print(f"False negatives (missed declines): {len(false_negatives):,}")
print(f"False positives (wrongly flagged): {len(false_positives):,}")
false_negatives[["content_hash_id", "impressions_month", "ctr_gap", "is_stale", "rf_prob"]].head(5)

False negatives (missed declines): 4,161
False positives (wrongly flagged): 3,397


,content_hash_id,impressions_month,ctr_gap,is_stale,rf_prob
1430,content_005605cd7de8938d,567.0,0.0000,0,0.270
1436,content_00efcc9c1c9bba3d,1860.0,0.0000,0,0.350
1438,content_011e8c2bdbe5375c,5566.0,0.0018,0,0.480
1442,content_01a313e966eb89cc,1310.0,0.0025,0,0.385
1443,content_01ce6214350f6733,771.0,0.0000,0,0.210


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.